In [4]:
import pandas as pd
import numpy as np

# 파일 불러오기
df = pd.read_excel("통합데이터.xlsx")

# -------------------------
# 숫자형 변환
# -------------------------
def to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", "").strip()
    if s == "":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

for col in ["면적", "고시면적", "토지면적", "출자면적"]:
    df[col] = df[col].apply(to_float)

# -------------------------
# 문자열 정리
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

for col in ["지목", "고시지목", "토지지목", "출자지목", "토지이동사유"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# -------------------------
# 지목 표준화
# -------------------------
def normalize_jimok(x):
    x = clean_text(x)

    mapping = {
        "대지": "대"
    }

    return mapping.get(x, x)

for col in ["지목", "고시지목", "토지지목", "출자지목"]:
    df[col] = df[col].apply(normalize_jimok)

# -------------------------
# 기준 1: 필지 합병되어 말소
# -------------------------
def check_merge_deleted(row):
    reason = row["토지이동사유"]
    if "필지 합병되어 말소" in reason:
        return 1
    return 0

df["필지합병말소"] = df.apply(check_merge_deleted, axis=1)

# -------------------------
# 기준 2: 면적다름
# - 단, 필지 합병되어 말소는 제외
# - 토지이음 면적이 고시/토지/출자 셋 중 하나라도 같으면 0
# - 셋 중 어디에도 없으면 1
# -------------------------
def check_area_diff(row):
    if row["필지합병말소"] == 1:
        return 0

    eum = row["면적"]
    g = row["고시면적"]
    t = row["토지면적"]
    c = row["출자면적"]

    if pd.notna(eum):
        match = (
            (pd.notna(g) and eum == g) or
            (pd.notna(t) and eum == t) or
            (pd.notna(c) and eum == c)
        )

        if not match:
            return 1

    return 0

df["면적다름"] = df.apply(check_area_diff, axis=1)

# -------------------------
# 기준 3: 지목다름
# - 단, 필지 합병되어 말소는 제외
# - 토지이음 지목이 고시/토지/출자 셋 중 하나라도 같으면 0
# - 셋 중 어디에도 없으면 1
# -------------------------
def check_jimok_diff(row):
    if row["필지합병말소"] == 1:
        return 0

    eum = row["지목"]
    g = row["고시지목"]
    t = row["토지지목"]
    c = row["출자지목"]

    if eum != "":
        match = (
            (g != "" and eum == g) or
            (t != "" and eum == t) or
            (c != "" and eum == c)
        )

        if not match:
            return 1

    return 0

df["지목다름"] = df.apply(check_jimok_diff, axis=1)

# -------------------------
# 최종 체크
# - 셋 중 하나라도 1이면 체크
# -------------------------
df["최종체크"] = (
    (df["면적다름"] == 1) |
    (df["지목다름"] == 1) |
    (df["필지합병말소"] == 1)
).astype(int)

# -------------------------
# 결과 사유
# -------------------------
def make_reason(row):
    reasons = []

    if row["면적다름"] == 1:
        reasons.append("면적다름")

    if row["지목다름"] == 1:
        reasons.append("지목다름")

    if row["필지합병말소"] == 1:
        reasons.append("필지 합병되어 말소")

    return ", ".join(reasons)

df["체크사유"] = df.apply(make_reason, axis=1)

# -------------------------
# 결과 정리
# -------------------------
result = df[[
    "최종체크",
    "체크사유",
    "면적다름",
    "지목다름",
    "필지합병말소",
    "소재지",
    "면적",
    "지목",
    "토지이동사유",
    "고시면적",
    "토지면적",
    "출자면적",
    "고시지목",
    "토지지목",
    "출자지목"
]].copy()

result = result.sort_values(
    ["최종체크", "면적다름", "지목다름", "필지합병말소", "소재지"],
    ascending=[False, False, False, False, True]
)

# 저장
result.to_excel("정부24_3기준_체크대상.xlsx", index=False)

print("완료")
print("최종 체크 대상 개수:", result["최종체크"].sum())
print("면적다름 개수:", result["면적다름"].sum())
print("지목다름 개수:", result["지목다름"].sum())
print("필지 합병되어 말소 개수:", result["필지합병말소"].sum())

display(result.head(30))

완료
최종 체크 대상 개수: 46
면적다름 개수: 20
지목다름 개수: 10
필지 합병되어 말소 개수: 17


,최종체크,체크사유,면적다름,지목다름,필지합병말소,소재지,면적,지목,토지이동사유,고시면적,토지면적,출자면적,고시지목,토지지목,출자지목
249,1,"면적다름, 지목다름",1,1,0,울산광역시 동구 방어동 1283-7,26732.0,공장용지,필지 합병,NaN,NaN,19111.0,,,잡종지
13,1,면적다름,1,0,0,울산광역시 남구 매암동 1-10,1582.0,공장용지,필지 분할,884.0,884.0,884.0,공장용지,공장용지,공장용지
24,1,면적다름,1,0,0,울산광역시 남구 매암동 139-29,46388.0,잡종지,필지 분할,49080.0,NaN,NaN,잡종지,,
25,1,면적다름,1,0,0,울산광역시 남구 매암동 139-30,1340.0,제방,필지 분할,1372.0,NaN,NaN,제방,,
48,1,면적다름,1,0,0,울산광역시 남구 매암동 139-82,1551.0,잡종지,필지 분할,2519.0,NaN,NaN,잡종지,,
89,1,면적다름,1,0,0,울산광역시 남구 매암동 472-42,2086.0,제방,면적정정,1800.0,1800.0,1800.0,제방,제방,제방
90,1,면적다름,1,0,0,울산광역시 남구 매암동 472-46,16261.0,잡종지,필지 분할,17486.0,45.0,NaN,잡종지,잡종지,
104,1,면적다름,1,0,0,울산광역시 남구 여천동 187-10,50780.0,잡종지,면적정정,49427.0,49427.0,49427.0,잡종지,잡종지,잡종지
105,1,면적다름,1,0,0,울산광역시 남구 여천동 187-12,114001.0,잡종지,면적정정,107564.0,107564.0,107564.0,잡종지,잡종지,잡종지
160,1,면적다름,1,0,0,울산광역시 남구 장생포동 144-35,5.0,대,필지 분할,131.0,NaN,NaN,대,,
